In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
   !pip install unsloth
else:
   import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
   xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
   !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
   !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
   !pip install --no-deps unsloth==2025.9.2
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [ ]:
r=16,
target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
lora_alpha=16,
lora_dropout=0,
bias="none",
use_gradient_checkpointing="unsloth",
random_state=42,
use_rslora=False,
loftq_config=None

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
df=pd.read_csv("Lead Scoring.csv")

In [ ]:
df.head(3)

,Prospect ID,Lead Number,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,...,Get updates on DM Content,Lead Profile,City,Asymmetrique Activity Index,Asymmetrique Profile Index,Asymmetrique Activity Score,Asymmetrique Profile Score,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.0,0,0.0,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Organic Search,No,No,0,5.0,674,2.5,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Email Opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,...,No,Potential Lead,Mumbai,02.Medium,01.High,14.0,20.0,No,Yes,Email Opened


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9240 entries, 0 to 9239
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Prospect ID                                    9240 non-null   str    
 1   Lead Number                                    9240 non-null   int64  
 2   Lead Origin                                    9240 non-null   str    
 3   Lead Source                                    9204 non-null   str    
 4   Do Not Email                                   9240 non-null   str    
 5   Do Not Call                                    9240 non-null   str    
 6   Converted                                      9240 non-null   int64  
 7   TotalVisits                                    9103 non-null   float64
 8   Total Time Spent on Website                    9240 non-null   int64  
 9   Page Views Per Visit                           9103 non-null   

In [ ]:
print(df["Converted"].value_counts())

Converted
0    5679
1    3561
Name: count, dtype: int64


2- Select relevant columns and handle missing data



In [ ]:
df.columns = df.columns.str.strip()

feature_cols = [
    'Lead Origin',
    'Lead Source',
    'Last Activity',
    'Country',
    'Specialization',
    'How did you hear about X Education',
    'What is your current occupation',
    'What matters most to you in choosing a course',
    'Tags'
]

target_col = 'Converted'


2.2 Drop rows with missing values for training purposes.


In [ ]:
all_cols = feature_cols + [target_col]
df_clean = df[all_cols].dropna().reset_index(drop=True)

In [ ]:
print(f"{len(df_clean)}")

4534


3- Use scikit-learn to split the DataFrame into three sets (80-10-10) with stratification.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['Converted'])
y = df_clean['Converted']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

Create conversational prompts for each lead

Define a function to format the lead data into prompts and another to generate conversations.
Apply these to each train / validation / test split.

In [ ]:
import pandas as pd

def format_lead_to_prompt(row, target_val=None):
    features_summary = ", ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])

    user_message = f"Analyze the following lead details and determine if they will convert:\n{features_summary}"

    if target_val is not None:
        assistant_message = "Yes, this lead is likely to convert." if target_val == 1 else "No, this lead is unlikely to convert."
        return {
            "messages": [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": assistant_message}
            ]
        }
    else:
        return {
            "messages": [
                {"role": "user", "content": user_message}
            ]
        }

def generate_conversations(X_split, y_split=None):
    conversations = []
    for idx in range(len(X_split)):
        row = X_split.iloc[idx]
        target = y_split.iloc[idx] if y_split is not None else None
        prompt_data = format_lead_to_prompt(row, target)
        conversations.append(prompt_data)
    return conversations

train_conversations = generate_conversations(X_train, y_train)
val_conversations = generate_conversations(X_val, y_val)
test_conversations = generate_conversations(X_test, y_test)

import json
print("--- Example Generated Conversation (Train) ---")
print(json.dumps(train_conversations[0], indent=2))

--- Example Generated Conversation (Train) ---
{
  "messages": [
    {
      "role": "user",
      "content": "Analyze the following lead details and determine if they will convert:\nLead Origin: API, Lead Source: Organic Search, Last Activity: SMS Sent, Country: India, Specialization: Select, How did you hear about X Education: Select, What is your current occupation: Unemployed, What matters most to you in choosing a course: Better Career Prospects, Tags: switched off"
    },
    {
      "role": "assistant",
      "content": "No, this lead is unlikely to convert."
    }
  ]
}


Convert conversations to Dataset objects with the Datasets library



In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_list(train_conversations)
val_dataset = Dataset.from_list(val_conversations)
test_dataset = Dataset.from_list(test_conversations)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 3627
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 453
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 454
    })
})


Apply the chat template and tokenize the data

Configure the tokenizer with the Llama 3.2 chat template and format prompts.
Tokenize the text for model input.

In [ ]:
!pip install --upgrade --force-reinstall huggingface_hub transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 57.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 33.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 65.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.9/808.9 kB 45.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.2 MB/s  0:00:00
  Attempting uninstall: xxhash
    Found existing installation: xxhash 4.0.2
    Uninst

In [ ]:
from huggingface_hub import login
login()


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
from google.colab import userdata
from transformers import AutoTokenizer

token = userdata.get('HF_TOKEN')

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)

print("Done")

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd

class LlamaInstructionDataset(Dataset):
    def __init__(self, X_split, y_split, tokenizer, max_length=512):
        self.encoded_texts = []

        for idx in range(len(X_split)):
            row = X_split.iloc[idx]
            target = y_split.iloc[idx] if y_split is not None else None

            features_summary = ", ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
            user_message = f"Analyze the following lead details and determine if they will convert:\n{features_summary}"

            messages = [{"role": "user", "content": user_message}]
            if target is not None:
                assistant_message = "Yes, this lead is likely to convert." if target == 1 else "No, this lead is unlikely to convert."
                messages.append({"role": "assistant", "content": assistant_message})

            formatted_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            encodings = tokenizer(
                formatted_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )

            self.encoded_texts.append(encodings["input_ids"].squeeze(0))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.encoded_texts)


Set the dataset format for PyTorch

Format the datasets with PyTorch-compatible columns.

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

train_dataset = LlamaInstructionDataset(X_train, y_train, tokenizer, max_length=512)
val_dataset = LlamaInstructionDataset(X_val, y_val, tokenizer, max_length=512)
test_dataset = LlamaInstructionDataset(X_test, y_test, tokenizer, max_length=512)

print("Done")
print(f"Train dataset length: {len(train_dataset)}")
print(f"Sample tensor shape: {train_dataset[0].shape}")

Done
Train dataset length: 3627
Sample tensor shape: torch.Size([512])
